# Agent 循环：观察、思考、行动

2026年的每个Agent————Claude Code、Cursor、Devin、Operator 都是2022年ReAct循环的变体。推理token间交错工具调用还有观察，直到到达某个终止条件。

## 问题描述

一个LLM自己只是一个自动补全。你问一个问题，得到一串字符串回复。它不能读文件、做查询、打开浏览器或者验证一项声明。如果模型有过时或者错误的信息，它会自信地说出错误的言论然后停止。

Agents通过一个模式修复它：一个让模型决定暂停、调用工具、读取结果、继续思考的循环。

## 基本概念

### ReAct 经典格式

`Reason + Act`， 每一轮：
```
Thought: ....
Action: ....
Observation: ....
Thought: ....
Action: ....
```

推理轨迹做到了只给动作的prompt做不到的三件事：制定一个计划、执行步时跟踪这个计划、当异常观测结果返回时处理异常。

### 2026年的转变：原生Reasoning

基于提示词的“思考”，也就是token，是2022年的临时方案。2025-2026年的API中将它们替换成了原生推理：模型在单独的管线中吐出推理上下文，这个管线在轮次中传递。

不变的是：循环本身。观察、思考、行动、观察、思考、行动、停止。不论思维token是打印在你的抄录中还是被单独的域承载，控制流都是相同的。

### 五个原料

每个agent训练都需要五个东西，缺了每一个就只是一个聊天机器人而不是一个agent。
1. 逐渐增长的会话历史。
2. 可以通过名字调用的工具注册。
3. 停止条件。   模型说停止、或者无工具调用、最大轮次限制、最大吞吐、或者护栏触发。
4. 轮次预算。  防止无限循环。
5. 观察格式器。  工具的输出需要转变成模型可以阅读的东西。

### 无处不在的循环

Claude Agent SDK、OpenAI Agents SDK、LangGraph... 每个底层运行的都是ReAct。框架的不同在于这层循环之上都有什么存在，状态检查点（LangGraph），角色-模型信息传递（AutoGen），角色模板（CrewAI）...

### 2026年的坑

- 信任边界崩塌。  模型的输出是不可信的输入。从网络上检索到的PDF可能包含隐藏的有害指令。
- 级联失败。  一个LLM幻觉或错误触发一连串下游错误/副作用，一错带多错。
- 循环长度爆炸。 2026年大多数的agent跑40～400步。要调试第38步的那个错误决策需要可观测性和评估轨迹。

# 开始编码

对应本章五个原料：**(1) 增长的历史 (2) 工具注册 (3) 停止条件 (4) 轮次预算 (5) 观察格式器**。先用脚本化 `ToyLLM` 看清控制流，再用 LangGraph + DeepSeek 跑真实 ReAct。


## 1. 教学玩具：ReAct 循环骨架


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Callable

from typing_extensions import TypedDict


@dataclass
class ToolCall:
    """一次工具调用请求。"""

    name: str
    args: dict[str, Any]


@dataclass
class Turn:
    """会话历史中的一步（用户 / 思考 / 行动+观察 / 终态）。"""

    kind: str
    content: str
    tool_call: ToolCall | None = None
    observation: str | None = None


class LLMReply(TypedDict, total=False):
    """脚本化 LLM 的一步输出（玩具）。"""

    kind: str  # "action" | "finish"
    content: str
    thought: str
    action: str
    args: dict[str, Any]


def format_observation(tool_name: str, raw: str, max_chars: int = 500) -> str:
    """
    观察格式器：把工具原始输出收成模型可读的短文本。

    Args:
        tool_name: 工具名。
        raw: ``dispatch`` 返回值。
        max_chars: 截断上限，防止观察淹没上下文。

    Returns:
        observation: 带标签的观察字符串。
    """
    text = raw if len(raw) <= max_chars else raw[: max_chars - 3] + "..."
    return f"[Observation from {tool_name}]\n{text}"


class ToolRegistry:
    """按名字分发的工具注册表。"""

    def __init__(self) -> None:
        self._tools: dict[str, Callable[..., str]] = {}

    def register(self, name: str, fn: Callable[..., str]) -> None:
        """
        Args:
            name: 工具名（LLM ``action`` 字段）。
            fn: 返回 ``str`` 的可调用对象。
        """
        self._tools[name] = fn

    def names(self) -> list[str]:
        """
        Returns:
            names: 已注册工具名（排序）。
        """
        return sorted(self._tools)

    def dispatch(self, call: ToolCall) -> str:
        """
        Args:
            call: 工具调用。

        Returns:
            result: 成功结果或 ``Error: ...`` 字符串（永不抛给循环外层）。
        """
        fn = self._tools.get(call.name)
        if fn is None:
            return f"Error: unknown tool '{call.name}'. Available: {self.names()}"
        try:
            return fn(**call.args)
        except TypeError as e:
            return f"Error: bad args for {call.name}: {call.args} ({e})"
        except Exception as e:
            return f"Error: {type(e).__name__}: {e}"


def calculator(expr: str) -> str:
    """
    安全计算器（白名单字符 + 无 builtins 的 ``eval``）。

    Args:
        expr: 算术表达式，如 ``"(3+5)*2"``。

    Returns:
        result: 计算结果或错误信息。
    """
    allowed = set("0123456789+-*/(). ")
    if not set(expr).issubset(allowed):
        return "Error: illegal characters in expr"
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))  # noqa: S307 — 玩具沙箱
    except Exception as e:
        return f"Error: {type(e).__name__}: {e}"


class KVStore:
    """进程内键值记忆（演示多工具协作）。"""

    def __init__(self) -> None:
        self._store: dict[str, str] = {}

    def get(self, key: str) -> str:
        """
        Args:
            key: 键。

        Returns:
            value: 值或 ``Missing: ...``。
        """
        return self._store.get(key, f"Missing: {key}")

    def set(self, key: str, value: str) -> str:
        """
        Args:
            key: 键。
            value: 值。

        Returns:
            ack: 确认字符串。
        """
        self._store[key] = value
        return f"Stored {key}={value}"


class ToyLLM:
    """
    脚本化 LLM：按预定剧本吐出 Thought/Action 或 finish。

    真正模型会读 ``history``；这里只推进游标，便于看清循环本身。
    """

    def __init__(self, script: list[LLMReply]) -> None:
        self.script = script
        self.cursor = 0

    def respond(self, history: list[Turn]) -> LLMReply:
        """
        Args:
            history: 当前会话（玩具未使用，保留接口对称）。

        Returns:
            reply: ``action`` 或 ``finish``。
        """
        _ = history
        if self.cursor >= len(self.script):
            return {"kind": "finish", "content": "(script exhausted)"}
        entry = self.script[self.cursor]
        self.cursor += 1
        return entry


@dataclass
class AgentLoop:
    """
    ReAct 控制流：历史增长 → 调 LLM → 工具 / 停止 → 写回观察。

    停止条件：``kind=finish`` 或 ``max_turns`` 预算耗尽。
    """

    llm: ToyLLM
    tools: ToolRegistry
    max_turns: int = 12
    history: list[Turn] = field(default_factory=list)

    def run(self, user_message: str) -> str:
        """
        Args:
            user_message: 用户问题。

        Returns:
            final: 终态文本（模型 finish 内容或 budget exhausted）。
        """
        self.history.append(Turn(kind="user", content=user_message))

        for _ in range(self.max_turns):
            reply = self.llm.respond(self.history)

            if reply.get("kind") == "finish":
                content = reply.get("content", "")
                self.history.append(Turn(kind="final", content=content))
                return content

            thought = reply.get("thought", "")
            if thought:
                self.history.append(Turn(kind="thought", content=thought))

            call = ToolCall(name=str(reply["action"]), args=dict(reply.get("args") or {}))
            raw = self.tools.dispatch(call)
            observation = format_observation(call.name, raw)
            self.history.append(
                Turn(
                    kind="action",
                    content=call.name,
                    tool_call=call,
                    observation=observation,
                )
            )

        self.history.append(Turn(kind="final", content="budget exhausted"))
        return "budget exhausted"


def print_trace(history: list[Turn]) -> None:
    """
    打印可读轨迹，便于对照「五个原料」。

    Args:
        history: ``AgentLoop.history``。
    """
    for i, t in enumerate(history):
        if t.kind == "user":
            print(f"{i:02d} USER        {t.content}")
        elif t.kind == "thought":
            print(f"{i:02d} THOUGHT     {t.content}")
        elif t.kind == "action":
            args = t.tool_call.args if t.tool_call else {}
            print(f"{i:02d} ACTION      {t.content}({args})")
            print(f"   OBS         {t.observation}")
        elif t.kind == "final":
            print(f"{i:02d} FINAL       {t.content}")
        else:
            print(f"{i:02d} {t.kind.upper():<11} {t.content}")


print("toy AgentLoop ready | tools protocol + observation formatter")


## 2. 玩具示例：多工具 + 停止条件 + 预算


In [ ]:
def demo_happy_path() -> None:
    """演示完整 ReAct：算式 → 写入 KV → 读回 → finish。"""
    kv = KVStore()
    registry = ToolRegistry()
    registry.register("calculator", calculator)
    registry.register("kv_set", kv.set)
    registry.register("kv_get", kv.get)

    script: list[LLMReply] = [
        {
            "kind": "action",
            "thought": "先算 (3+5)*7",
            "action": "calculator",
            "args": {"expr": "(3+5)*7"},
        },
        {
            "kind": "action",
            "thought": "把结果存到 answer",
            "action": "kv_set",
            "args": {"key": "answer", "value": "56"},
        },
        {
            "kind": "action",
            "thought": "读回校验",
            "action": "kv_get",
            "args": {"key": "answer"},
        },
        {
            "kind": "finish",
            "content": "结果是 56，已写入 answer。",
        },
    ]
    agent = AgentLoop(llm=ToyLLM(script), tools=registry, max_turns=8)
    final = agent.run("请计算 (3+5)*7，存到 answer，再读出来确认。")
    print("=== happy path ===")
    print_trace(agent.history)
    print("final =", final)
    assert final.startswith("结果是")
    assert kv.get("answer") == "56"


def demo_unknown_tool_and_budget() -> None:
    """演示工具错误观察 + 轮次预算耗尽（无 finish）。"""
    registry = ToolRegistry()
    registry.register("calculator", calculator)
    script: list[LLMReply] = [
        {
            "kind": "action",
            "thought": "误用未注册工具",
            "action": "web_search",
            "args": {"q": "pi"},
        },
        {
            "kind": "action",
            "thought": "改用计算器",
            "action": "calculator",
            "args": {"expr": "22/7"},
        },
        # 故意不 finish，逼出 budget
        {
            "kind": "action",
            "thought": "继续空转",
            "action": "calculator",
            "args": {"expr": "1+1"},
        },
    ]
    agent = AgentLoop(llm=ToyLLM(script), tools=registry, max_turns=2)
    final = agent.run("随便算点东西")
    print("\n=== error obs + budget ===")
    print_trace(agent.history)
    print("final =", final)
    assert final == "budget exhausted"
    assert "unknown tool" in (agent.history[2].observation or "")


demo_happy_path()
demo_unknown_tool_and_budget()
print("\nTOY DEMOS OK")


## 3. 生产级：LangGraph（`create_agent`）+ DeepSeek

底层仍是同一套 ReAct；框架负责消息状态、原生 tool_calls、以及可扩展的 checkpointer。
需已配置项目根目录 `.env` 中的 ``DEEPSEEK_API_KEY``。


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

# sources/15_Agent/01_Agent_Loop → ../../00_Common
sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"


@tool
def calculator_tool(expr: str) -> str:
    """Evaluate a simple arithmetic expression like '(3+5)*7'."""
    return calculator(expr)


_kv = KVStore()


@tool
def kv_set_tool(key: str, value: str) -> str:
    """Store a string value under key in the session KV store."""
    return _kv.set(key, value)


@tool
def kv_get_tool(key: str) -> str:
    """Read a string value by key from the session KV store."""
    return _kv.get(key)


def build_deepseek_react_agent(
    *,
    model: str = MODEL,
    max_turns_hint: int = 12,
) -> Any:
    """
    构造生产 ReAct agent（LangGraph CompiledStateGraph）。

    Args:
        model: LangChain 模型 id，默认 DeepSeek flash。
        max_turns_hint: 写入 system prompt 的软轮次提示（硬限制由模型/运行时行为约束）。

    Returns:
        agent: 可 ``invoke`` / ``stream`` 的图。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")

    llm = init_chat_model(
        model,
        temperature=0,
        # DeepSeek V4 若默认开 thinking，部分工具链路会更慢；教学关闭
        extra_body={"thinking": {"type": "disabled"}},
    )
    return create_agent(
        llm,
        tools=[calculator_tool, kv_set_tool, kv_get_tool],
        system_prompt=(
            "You are a careful tool-using agent. "
            "Use calculator_tool for math, kv_set_tool/kv_get_tool for memory. "
            f"Prefer finishing within ~{max_turns_hint} tool rounds. "
            "After tools succeed, give a short final answer in Chinese."
        ),
    )


def format_agent_messages(messages: list[BaseMessage]) -> str:
    """
    把 LangGraph 轨迹格式化成接近玩具 ``print_trace`` 的文本。

    Args:
        messages: ``agent.invoke(... )['messages']``。

    Returns:
        trace: 多行可读轨迹。
    """
    lines: list[str] = []
    for i, m in enumerate(messages):
        if isinstance(m, HumanMessage):
            lines.append(f"{i:02d} USER        {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(
                        f"{i:02d} ACTION      {tc.get('name')}({tc.get('args')})"
                    )
            if m.content:
                lines.append(f"{i:02d} ASSISTANT   {m.content}")
        elif isinstance(m, ToolMessage):
            lines.append(f"{i:02d} OBS         [{m.name}] {m.content}")
        else:
            lines.append(f"{i:02d} {type(m).__name__:<11} {getattr(m, 'content', m)}")
    return "\n".join(lines)


def run_deepseek_agent(question: str) -> dict[str, Any]:
    """
    Args:
        question: 用户问题。

    Returns:
        result: ``create_agent`` 的 invoke 结果（含 ``messages``）。
    """
    agent = build_deepseek_react_agent()
    return agent.invoke({"messages": [HumanMessage(content=question)]})


print("production helpers ready |", MODEL)


## 4. 生产示例：调用 DeepSeek 跑一轮真实循环


In [ ]:
def demo_deepseek_react() -> None:
    """需要网络与有效 ``DEEPSEEK_API_KEY``。"""
    question = "请计算 (3+5)*7，把结果存到 key=answer，再读出来确认，最后用中文简短回答。"
    result = run_deepseek_agent(question)
    messages = result["messages"]
    print("=== DeepSeek + LangGraph ReAct ===")
    print(format_agent_messages(messages))
    last = messages[-1]
    print("\nlast content:", getattr(last, "content", last))
    # 工具侧效应：若模型按要求写了 KV，应能读到
    print("kv answer =", _kv.get("answer"))


demo_deepseek_react()
